# Day 3 — Swing-Up Control

## The Problem LQR Can't Solve

The LQR controller built in Day 2 is a **local** controller. It works beautifully when the pole starts within ~20° of upright, because that's the region where the linearized model is valid. Start the pole hanging down and LQR is completely useless — the linearization breaks down, and the controller either does nothing or actively destabilises the system.

This is the swing-up problem: get the pole from **any arbitrary state** to the upright equilibrium, at which point LQR takes over and balances it.

```
Any state  →  [Swing-Up: energy pumping]  →  |θ| < 20°  →  [LQR: balance]
```

---

## Phase Space: What LQR Actually Does to the System

To understand why swing-up is needed, it helps to look at what the system's phase space looks like with and without control.

**Open loop (LQR off):**

In the $(\theta, \dot{\theta})$ plane, there are two equilibrium points:

- $\theta = 0$ — pole hanging down. **Stable**. All nearby trajectories spiral into it. This is the natural attractor.
- $\theta = \pi$ — pole upright. **Unstable**. A saddle point. Any tiny perturbation causes the system to run away.

The upright point has no basin of attraction — it repels trajectories rather than attracting them.

**Closed loop (LQR on):**

The controller adds a state-dependent force $F = -Kx$, which **reshapes the vector field** around $\theta = \pi$. The upright point's stability character fundamentally changes. The eigenvalues of $A - BK$ are all in the left half-plane, meaning the upright point becomes **locally asymptotically stable** — a genuine attractor.

The hanging equilibrium $\theta = 0$ does not disappear. It still exists in the nonlinear system. What LQR creates is a **finite region** around $\theta = \pi$ which is called the **region of attraction** — where the upright point wins:

$$\mathcal{R} = \{x_0 : \lim_{t \to \infty} x(t) = x^* \text{ under LQR}\}$$

Outside $\mathcal{R}$, the linearization is no longer valid and LQR loses its stabilising power. The hanging equilibrium can still pull the system in. This is why the region of attraction is **bounded** — LQR does not make the upright point globally stable.

Swing-up's job is singular: push the state from wherever it is **into** $\mathcal{R}$, at which point LQR takes over and the upright attractor captures it.

---

## The Core Idea — Energy Pumping

The upright position has a specific total mechanical energy $E^*$. The controller compares the **current energy** $E$ to $E^*$ and pushes the cart in whatever direction adds energy to the system at that moment. As $E \rightarrow E^*$, the pole naturally rises toward upright.

---

## The Energy Function — Full Derivation

### 1. Setting Up Coordinates

> **Convention note:** Throughout this document, $\theta = 0$ is the hanging (downward) position and $\theta = \pi$ is the upright position. This follows the convention in Åström & Furuta (1996). The simulation code uses the opposite convention ($\theta = 0$ upright), so a coordinate shift of $\pi$ is applied when connecting theory to code.

Define the system using two coordinates: $x$ (cart position) and $\theta$ (pole angle from downward vertical, so $\theta = \pi$ is upright).

The position of the pendulum bob is:

$$x_p = x + L\sin(\theta), \qquad y_p = -L\cos(\theta)$$

Taking time derivatives gives the velocities of the bob:

$$\dot{x}_p = \dot{x} + L\dot{\theta}\cos(\theta), \qquad \dot{y}_p = L\dot{\theta}\sin(\theta)$$

### 2. Total Energy of the Pendulum

The total mechanical energy of the pendulum bob (mass $m$) is:

$$E = \frac{1}{2}m(\dot{x}_p^2 + \dot{y}_p^2) + mgy_p$$

Expanding the velocity squared term:

$$\dot{x}_p^2 + \dot{y}_p^2 = \left(\dot{x} + L\dot{\theta}\cos\theta\right)^2 + \left(L\dot{\theta}\sin\theta\right)^2 = \dot{x}^2 + 2\dot{x}L\dot{\theta}\cos\theta + L^2\dot{\theta}^2$$

So the full energy expression is:

$$E = \frac{1}{2}m\dot{x}^2 + m\dot{x}L\dot{\theta}\cos(\theta) + \frac{1}{2}mL^2\dot{\theta}^2 - mgL\cos(\theta)$$

### 3. Computing $\dot{E}$

Taking the total time derivative using the chain rule across all variables:

$$\dot{E} = \frac{\partial E}{\partial \dot{x}}\ddot{x} + \frac{\partial E}{\partial \theta}\dot{\theta} + \frac{\partial E}{\partial \dot{\theta}}\ddot{\theta}$$

When you compute these partial derivatives and simplify using the full equations of motion (gravity and inertial terms cancel internal energy transfers), a clean result emerges:

$$\boxed{\dot{E} = mL\ddot{x}\,\dot{\theta}\cos(\theta)}$$

**Physical intuition:** If the cart isn't accelerating ($\ddot{x} = 0$), it acts as an inertial frame and the pendulum's energy is conserved. Energy only flows into the pendulum when the cart *accelerates*, pushing or pulling on the pivot point.

### 4. Connecting $\ddot{x}$ to the Force $F$

From Newton's second law applied to the total system mass along the horizontal axis:

$$F \approx (M + m)\ddot{x} \implies \ddot{x} \approx \frac{F}{M + m}$$

Substituting back:

$$\dot{E} = mL \cdot \frac{F}{M+m} \cdot \dot{\theta}\cos(\theta) = F \cdot L\dot{\theta}\cos(\theta) \cdot \frac{m}{M+m}$$

This is the key relationship: **the rate of change of pendulum energy is directly proportional to $F \cdot \dot{\theta} \cdot \cos(\theta)$**.

---

## Deriving the Control Law via Lyapunov

Now that we know how energy flows, we can design a controller that deliberately drives $E \rightarrow E^*$.

### The Lyapunov Function

Define a candidate Lyapunov function — a scalar measure of "how far we are from the energy target":

$$V = \frac{1}{2}(E - E^*)^2$$

This is always non-negative and equals zero only when $E = E^*$. Taking its time derivative:

$$\dot{V} = (E - E^*) \cdot \dot{E}$$

For the system to converge to $E^*$, we need $\dot{V} \leq 0$ — the Lyapunov condition. This means $(E - E^*)$ and $\dot{E}$ must have **opposite signs**.

### Choosing $F$ to Satisfy the Condition

Substituting our expression for $\dot{E}$:

$$\dot{V} = (E - E^*) \cdot F \cdot L\dot{\theta}\cos(\theta) \cdot \frac{m}{M+m}$$

To make this negative, we choose:

$$F = k \cdot (E - E^*) \cdot \dot{\theta} \cdot \cos(\theta), \qquad k < 0$$

This gives:

$$\dot{V} = k \cdot (E - E^*)^2 \cdot \dot{\theta}^2\cos^2(\theta) \cdot \frac{mL}{M+m} \leq 0$$

The control law was not guessed — it was **reverse-engineered** from the requirement that $\dot{V} \leq 0$. This is Lyapunov-based control.

### The Smooth Control Law

$$\boxed{F = k \cdot (E - E^*) \cdot \dot{\theta} \cdot \cos(\theta), \quad k < 0}$$

| Term | Role |
|---|---|
| $E - E^*$ | Energy deficit — how hard to push and in which sense |
| $\dot{\theta}$ | Angular velocity — timing of the push |
| $\cos(\theta)$ | Coupling — cart force most effective near hanging position |
| $k$ | Gain — must be negative to satisfy Lyapunov condition |

The target energy at the upright equilibrium $(\theta = \pi,\ \dot{\theta} = 0)$:

$$E^* = \frac{1}{2}mL^2(0)^2 - mgL\cos(\pi) = mgL$$

---

## The Problem With the Smooth Law — and the Bang-Bang Fix

### Why the Smooth Law Stalls at Rest

The smooth law has a critical weakness: its force is proportional to $\dot{\theta}$. At exact rest ($\dot{\theta} = 0$), the force is zero regardless of how large the energy error is:

$$F = k \cdot (E - E^*) \cdot \underbrace{0}_{\dot{\theta}} \cdot \cos(\theta) = 0$$

This is not a numerical glitch. It is a mathematical consequence of the formula. The controller has no mechanism to deliver a first kick. In a real physical system, there is always noise; in simulation, floating-point values are close to zero but never exactly zero — but the resulting force is so small that other effects (wall friction, numerical damping) suppress it entirely. The pendulum sits there indefinitely.

### Åström's Solution — the Bang-Bang Law

Åström & Furuta (1996, Eq. 4) replace the raw $\dot{\theta}\cos\theta$ with its sign:

$$\boxed{F = \text{sat}\!\left(k\,|E - E^*|\right) \cdot \text{sign}(-\dot{\theta}\cos\theta), \quad k > 0}$$

where $\text{sat}$ clips the output to $\pm F_{\max}$. Here the sign() is the signum function

The key difference between the two laws:

| Smooth law | Bang-bang law |
|---|---|
| $\|F\| \propto \|\dot{\theta}\cos\theta\|$ | $\|F\|$ depends only on energy error |
| $\dot{\theta} \to 0 \Rightarrow F \to 0$ | $\dot{\theta} \to 0^{\pm} \Rightarrow F \to \pm F_{\text{err}}$ |
| Slow start, stalls at rest | Commits immediately, full force direction |
| Proportional control | Approaches bang-bang (time-optimal) |

**Why bang-bang?** Pontryagin's Minimum Principle states that for systems with bounded inputs, the time-optimal control is bang-bang — always at the boundary $\pm F_{\max}$, switching at critical moments. Åström notes this explicitly: *"minimum time strategies for swinging up the pendulum are of bang-bang type."* The sign-based law is an approximation to this optimal strategy, with magnitude tapered near $E = E^*$ to avoid over-shooting.

**$\text{sign}(0)$ is still zero.** The bang-bang law does not self-start from perfect rest — sign of zero is zero. It is far more robust near rest (any infinitesimal $\dot{\theta}$ gives full force), but a deliberate initial perturbation is still required to break the symmetry. In the simulation, pressing `S` adds a small $\dot{\theta} = 0.05\ \text{rad/s}$. In Åström §4: *"Accelerate with maximum acceleration in an arbitrary direction"* — this is the same idea.

---

## The Physical Maneuver — Step by Step

This section traces exactly what happens from the moment swing-up is activated.

**Initial state:** $\theta = 0$ (hanging), $\dot{\theta} = 0$. After pressing `S`, a tiny kick gives $\dot{\theta} = +\varepsilon$.

**Step 1 — First half-swing.**
At $\theta \approx 0$, $\cos\theta \approx 1$, $E - E^* \approx -2mgL < 0$.
The force law gives $F > 0$ — cart moves right. The bob is also moving in the same direction. The cart's acceleration adds energy to the pendulum (the cart pulls the pivot away from the bob, increasing the angular momentum).

**Step 2 — Bob rises, reaches max angle.**
$\dot{\theta}$ decreases as the bob rises. At the turning point $\dot{\theta} = 0$. No force for an instant. Bob reverses.

**Step 3 — Return half-swing.**
Now $\dot{\theta} < 0$. The sign flips. Cart now moves left — again in the same direction as the bob's motion. Energy is added on the return swing too.

**Step 4 — Passing through horizontal ($\theta = \pi/2$).**
$\cos(\pi/2) = 0$. Force is exactly zero. Controllability is lost momentarily. This is a fundamental limitation — Åström §3: *"controllability is lost when the pendulum is horizontal."* Momentum carries the bob through without any help.

**Step 5 — Energy accumulates each cycle.**
Each full oscillation adds a net positive amount of energy. The amplitude grows. This is the pumping-a-swing mechanism.

**Step 6 — Approaching $E = E^*$.**
As $E \rightarrow E^*$, the magnitude $k|E - E^*|$ shrinks. The cart pushes more gently. The pendulum coasts toward upright.

**Step 7 — Handoff.**
When $|\theta_{\text{wrapped}}| < 20°$ **and** $|\dot{\theta}| < 3.0\ \text{rad/s}$, control switches to LQR. The velocity condition is critical — if the pole is flying through upright at high speed, LQR cannot grab it.

---

## Why Any Initial State Works

This is the key advantage of energy control over position-based control. The law operates entirely in *energy space*, not *configuration space*. It does not care how the state got to its current value — only what the current energy is versus the target.

| Initial state | $E - E^*$ | Behaviour |
|---|---|---|
| Hanging at rest (kicked) | $\approx -2mgL$ | Large deficit — pumps hard, builds energy from scratch |
| Mid-swing, small amplitude | Moderate negative | Continues pumping, growing each cycle |
| Mid-swing, large amplitude | Small negative | Gentle top-up |
| At target energy | $\approx 0$ | Force near zero — momentum-driven coast to upright |
| **Over-energised (spinning past upright)** | **Positive** | **Force reverses — energy is actively removed** |
| Near upright, slow | $\approx 0$ | Handoff to LQR fires |

The over-energised case is the most important: **the same control law removes excess energy.** If the cart pumps too hard and the pole spins past the top with too much speed, $E > E^*$ and $E - E^* > 0$. The force direction automatically reverses, taking energy away until $E$ comes back down to $E^*$. No separate logic is needed.

The Lyapunov function $V = \frac{1}{2}(E - E^*)^2$ is symmetric around the target — it is positive on *both* sides. The gradient always points toward $E^*$.

---

## Edge Cases and Their Handling

| Case | What happens | How it's handled |
|---|---|---|
| **Perfect rest** $(\dot{\theta} = 0)$ | $F = 0$ — no self-start | Press `S` adds tiny $\dot{\theta}$ kick |
| **Exact horizontal** $(\cos\theta = 0)$ | $F = 0$ — brief controllability loss | Momentum carries through; natural |
| **Fast passage through threshold** | LQR velocity condition not met | Velocity check in `should_handoff()` prevents premature handoff |
| **Cart hits wall** | Position penalty forces return | Wall penalty term added to $F$ |
| **Over-pump** $(E \gg E^*)$ | Pole spins continuously | Controller auto-removes energy; reduce $k$ if persistent |

---

## The Handoff

When the pole enters the LQR capture region, control switches instantly:

```python
if should_handoff(state):          # |θ_wrapped| < 20° AND |θ̇| < 3.0 rad/s
    switch to LQR
else:
    F = swing_up_force(state)
```

Two conditions are checked, not one:

- **Angle condition** $|\theta_{\text{wrapped}}| < 20°$: pole is geometrically near upright.
- **Velocity condition** $|\dot{\theta}| < 3.0$: pole is arriving slowly, not flying through.

LQR can only stabilise states inside its region of attraction. A pole crossing the $20°$ threshold at high angular velocity is almost certainly outside $\mathcal{R}$ — it needs more energy dissipation before LQR can hold it.

---

## Summary of Key Concepts

| Concept | What it means here |
|---|---|
| **Lyapunov-based control** | Design $F$ so that a chosen energy-like function $V$ always decreases |
| **Energy shaping** | Control the system's total energy rather than position or velocity directly |
| **Bang-bang control** | Force always at its maximum magnitude; only direction varies |
| **Pontryagin's principle** | Time-optimal bounded-input control is bang-bang; sign() law approximates this |
| **Region of attraction** | The finite set of states from which LQR successfully stabilises the upright point |
| **Hybrid control** | Two controllers with a switching condition — swing-up drives into $\mathcal{R}$, LQR stabilises inside it |

---

## References

1. **Åström, K. J., & Furuta, K. (2000).** *Swinging up a pendulum by energy control.* Automatica, 36(2), 287–295. — Definitive reference. The control law, Pontryagin connection, and multi-swing analysis all come from here.

2. **Fradkov, A. L. (1991).** *Speed-gradient laws of control and evolution.* Proc. 1st European Control Conference. — Foundational work on the speed-gradient method, the general framework that energy-shaping swing-up is a special case of.

3. **Spong, M. W. (1995).** *The swing up control problem for the acrobot.* IEEE Control Systems Magazine, 15(1), 49–55. — Complementary approach using partial feedback linearization.

4. **Khalil, H. K. (2002).** *Nonlinear Systems* (3rd ed.), Ch. 4. Prentice Hall. — Standard reference for Lyapunov stability theory, LaSalle's invariance principle, and region of attraction analysis.

5. **Fantoni, I., & Lozano, R. (2002).** *Non-linear Control for Underactuated Mechanical Systems.* Springer. — Book-length treatment of cart-pole, acrobot, and related underactuated systems.